# Deploy SEI_Dashboard to this Fabric workspace

Creates the **SEI_Dashboard** semantic model and the 9-page **SEI_Dashboard** report in the workspace this notebook lives in, from the files in the public GitHub repo. No Git integration, Azure DevOps or Power BI Desktop needed.

**Run the cells top to bottom** (or *Run all*). Takes about 2-3 minutes.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import time
import requests
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.report import create_report_from_reportjson

REPO = "https://raw.githubusercontent.com/NirmalKumar31/social-engagement-intelligence-pro/main/powerbi"
NAME = "SEI_Dashboard"

bim = requests.get(f"{REPO}/SEI_Dashboard.SemanticModel/model.bim", timeout=60).json()
report_json = requests.get(f"{REPO}/SEI_Dashboard.Report/report.json", timeout=60).json()
print(f"Loaded model: {len(bim['model']['tables'])} tables, {len(bim['model']['relationships'])} relationships")
print(f"Loaded report: {len(report_json['sections'])} pages")

## 1. Create the semantic model

In [ ]:
datasets = fabric.list_datasets(mode="rest")
if NAME in datasets["Dataset Name"].values:
    print(f"'{NAME}' semantic model already exists, skipping")
else:
    labs.create_semantic_model_from_bim(dataset=NAME, bim_file=bim)

# wait until the service lists it
for _ in range(30):
    if NAME in fabric.list_datasets(mode="rest")["Dataset Name"].values:
        print("semantic model is ready")
        break
    time.sleep(5)

## 2. Create the report

In [ ]:
create_report_from_reportjson(report=NAME, dataset=NAME, report_json=report_json)

## 3. Load the data

If this cell fails with a **credentials** error, that's expected the first time:

1. In the workspace, hover the **SEI_Dashboard** semantic model, then **... → Settings → Data source credentials → Edit credentials**.
2. Choose **Anonymous**, privacy level **Public**, then **Sign in**.
3. Re-run this cell (or use **... → Refresh now** on the semantic model).

In [ ]:
labs.refresh_semantic_model(dataset=NAME)
print("Done. Open the SEI_Dashboard report in this workspace.")